In [0]:
MY_ID = "2328073"

CATALOG = "workspace"
SCHEMA = "dbsf_2328073"
VOL = "/Volumes/workspace/dbsf_2328073/raw_files"

print("MY_ID:", MY_ID)
print("CATALOG:", CATALOG)
print("SCHEMA:", SCHEMA)
print("VOL:", VOL)

MY_ID: 2328073
CATALOG: workspace
SCHEMA: dbsf_2328073
VOL: /Volumes/workspace/dbsf_2328073/raw_files


In [0]:
display(dbutils.fs.ls(VOL))


path,name,size,modificationTime
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/bronze/,bronze/,0,1790744330902
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/gold_export/,gold_export/,0,1790744330902
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_customers_dataset.csv,olist_customers_dataset.csv,9033957,1790694849000
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_geolocation_dataset.csv,olist_geolocation_dataset.csv,61273883,1790694867000
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15438671,1790694852000
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5777138,1790694848000
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_order_reviews_dataset.csv,olist_order_reviews_dataset.csv,14451670,1790694851000
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1790694852000
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_products_dataset.csv,olist_products_dataset.csv,2379446,1790694845000
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_sellers_dataset.csv,olist_sellers_dataset.csv,174703,1790694843000


In [0]:
for f in dbutils.fs.ls(VOL):
    print(f.name)

bronze/
gold_export/
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv
sales_json/
sales_parquet/


In [0]:
from pyspark.sql import functions as F

FILES = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
    "geolocation": "olist_geolocation_dataset.csv"
}

def load_bronze(table_name, file_name):

    print(f"\nLoading: {file_name}")

    # Read the CSV as raw strings
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(f"{VOL}/{file_name}")
    )

    # Save original source columns
    raw_columns = df.columns

    # Unity Catalog-compatible file metadata
    df = (
        df
        .withColumn(
            "_source_file",
            F.col("_metadata.file_path")
        )
        .withColumn(
            "_ingested_at",
            F.current_timestamp()
        )
        .withColumn(
            "_row_hash",
            F.sha2(
                F.concat_ws(
                    "||",
                    *[
                        F.coalesce(
                            F.col(c).cast("string"),
                            F.lit("")
                        )
                        for c in raw_columns
                    ]
                ),
                256
            )
        )
    )

    target_table = (
        f"{CATALOG}.{SCHEMA}.bronze_{table_name}_2328073"
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    row_count = df.count()

    print(f"Created: {target_table}")
    print(f"Rows: {row_count:,}")


for table_name, file_name in FILES.items():
    load_bronze(table_name, file_name)

print("\n==============================")
print("BRONZE INGESTION COMPLETED")
print("==============================")


Loading: olist_orders_dataset.csv
Created: workspace.dbsf_2328073.bronze_orders_2328073
Rows: 99,441

Loading: olist_order_items_dataset.csv
Created: workspace.dbsf_2328073.bronze_order_items_2328073
Rows: 112,650

Loading: olist_order_payments_dataset.csv
Created: workspace.dbsf_2328073.bronze_order_payments_2328073
Rows: 103,886

Loading: olist_order_reviews_dataset.csv
Created: workspace.dbsf_2328073.bronze_order_reviews_2328073
Rows: 104,162

Loading: olist_customers_dataset.csv
Created: workspace.dbsf_2328073.bronze_customers_2328073
Rows: 99,441

Loading: olist_products_dataset.csv
Created: workspace.dbsf_2328073.bronze_products_2328073
Rows: 32,951

Loading: olist_sellers_dataset.csv
Created: workspace.dbsf_2328073.bronze_sellers_2328073
Rows: 3,095

Loading: product_category_name_translation.csv
Created: workspace.dbsf_2328073.bronze_category_translation_2328073
Rows: 71

Loading: olist_geolocation_dataset.csv
Created: workspace.dbsf_2328073.bronze_geolocation_2328073
Rows: 1,

In [0]:
bronze_tables = [
    "bronze_orders_2328073",
    "bronze_order_items_2328073",
    "bronze_order_payments_2328073",
    "bronze_order_reviews_2328073",
    "bronze_customers_2328073",
    "bronze_products_2328073",
    "bronze_sellers_2328073",
    "bronze_category_translation_2328073",
    "bronze_geolocation_2328073"
]

print("====================================")
print("BRONZE TABLE VALIDATION")
print("====================================")

for table in bronze_tables:
    full_name = f"{CATALOG}.{SCHEMA}.{table}"
    count = spark.table(full_name).count()
    print(f"{table:<45} {count:,}")

BRONZE TABLE VALIDATION
bronze_orders_2328073                         99,441
bronze_order_items_2328073                    112,650
bronze_order_payments_2328073                 103,886
bronze_order_reviews_2328073                  104,162
bronze_customers_2328073                      99,441
bronze_products_2328073                       32,951
bronze_sellers_2328073                        3,095
bronze_category_translation_2328073           71
bronze_geolocation_2328073                    1,000,163


In [0]:
orders_bronze = spark.table(
    "workspace.dbsf_2328073.bronze_orders_2328073"
)

print("Columns:")
print(orders_bronze.columns)

display(orders_bronze.limit(10))

Columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', '_source_file', '_ingested_at', '_row_hash']


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,_source_file,_ingested_at,_row_hash
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_orders_dataset.csv,2026-09-30T05:02:11.036Z,190c908245b605074be9a9550988eb25e067e071b19e38fbb8c36b4b7cd2aa49
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_orders_dataset.csv,2026-09-30T05:02:11.036Z,336c0fd204b2b1650777128758969e43f15c94d717b7a37f93bf11069d9ff26e
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_orders_dataset.csv,2026-09-30T05:02:11.036Z,d51a2073c7e4b9ff1ef36ae973675c620bfb29f3be7bc0079d80ace28dded954
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_orders_dataset.csv,2026-09-30T05:02:11.036Z,711afa09807ef411a7a2714034a4cb35a3f8071b29c3a7773766c8d749e73f1b
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_orders_dataset.csv,2026-09-30T05:02:11.036Z,b740b0ee10037883f787945be93fffc1e358d0717c8ee84c1ddee814efaa2819
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00,dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_orders_dataset.csv,2026-09-30T05:02:11.036Z,752f0c95b8ee594b7dd4eeb7fa5521e7cc5dc444a3b2917e547f6393894f7f0c
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,null,null,2017-05-09 00:00:00,dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_orders_dataset.csv,2026-09-30T05:02:11.036Z,5da7ec019a6d7f29a67978a3fd34c3bc80594b346f985f4666400436c7753d71
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00,dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_orders_dataset.csv,2026-09-30T05:02:11.036Z,7b0d39489b90859fa550de12cfab6bec4e27f30429caa6d6f53dbab2707f2558
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00,dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_orders_dataset.csv,2026-09-30T05:02:11.036Z,de222a36bd1285cb9332b7c37290a044cd3b13d511424062d8d3963119cf90b5
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00,dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_orders_dataset.csv,2026-09-30T05:02:11.036Z,21123e937f4f8a330a6d80fae36cbeda490465280cab26a08b0a60d303f477f9


In [0]:
display(
    spark.sql("""
        SELECT
            _source_file,
            COUNT(*) AS row_count
        FROM workspace.dbsf_2328073.bronze_orders_2328073
        GROUP BY _source_file
    """)
)

_source_file,row_count
dbfs:/Volumes/workspace/dbsf_2328073/raw_files/olist_orders_dataset.csv,99441
